In [ ]:
import os
import torch
import jiwer
import librosa
import pandas as pd
from transformers import pipeline
from tqdm import tqdm

def normalize_text(text):
    transform = jiwer.Compose([
        jiwer.ToLowerCase(),
        jiwer.RemovePunctuation(),
        jiwer.RemoveMultipleSpaces(),
        jiwer.Strip(),
    ])
    return transform(str(text))

def main():
    tsv_path = "../voice_dataset/az/test.tsv"
    clips_dir = "../voice_dataset/az/clips/"

    df = pd.read_csv(tsv_path, sep='\t')
    
    text_col = "text" if "text" in df.columns else "sentence"

    # complete file path
    df['full_audio_path'] = df['path'].apply(lambda x: os.path.join(clips_dir, x))

    # check if gpu exist
    device = "cuda:0" if torch.cuda.is_available() else "cpu"

    asr_pipeline = pipeline(
        "automatic-speech-recognition",
        model="openai/whisper-small",
        torch_dtype=torch.float32,
        device=device,
        model_kwargs={"attn_implementation": "sdpa"} if torch.cuda.is_available() else {}
    )

    # tell the model to use azerbaijani langauge and not to translate to english
    asr_pipeline.model.config.forced_decoder_ids = asr_pipeline.tokenizer.get_decoder_prompt_ids(language="az", task="transcribe")

    results = []

    # tqdm is just for seeing the progress
    for index, row in tqdm(df.iterrows(), total=len(df), desc="Transcribing Test Split"):
        reference_raw = str(row[text_col])
        audio_file = row["full_audio_path"]

        try:
            audio_array, _ = librosa.load(audio_file, sr=16000)
        except Exception:
            continue

        prediction_raw = asr_pipeline(audio_array)["text"]

        reference_clean = normalize_text(reference_raw)
        prediction_clean = normalize_text(prediction_raw)

        if len(reference_clean) == 0:
            continue

        wer = jiwer.wer(reference_clean, prediction_clean)
        cer = jiwer.cer(reference_clean, prediction_clean)

        results.append({
            "Reference": reference_raw,
            "Prediction": prediction_raw,
            "WER": wer,
            "CER": cer
        })

    results_df = pd.DataFrame(results)

    avg_wer = results_df["WER"].mean() * 100
    avg_cer = results_df["CER"].mean() * 100

    # get the best and worst cases
    df_sorted = results_df.sort_values(by="WER")
    best_5 = df_sorted.head(5)
    worst_5 = df_sorted.tail(5)

    # printing
    print("\n--- ASR Performance Metrics ---")
    print(f"Total Test Clips Evaluated: {len(results_df)}")
    print(f"Average WER: {avg_wer:.2f}%")
    print(f"Average CER: {avg_cer:.2f}%\n")

    print("--- Top 5 Best Predictions ---")
    print(best_5.to_string(index=False))
    print("\n--- Top 5 Worst Predictions ---")
    print(worst_5.to_string(index=False))

if __name__ == "__main__":
    main()